In [4]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
gen=pd.read_csv('gender_submission.csv')

In [8]:
gen.shape

(418, 2)

In [11]:
df= pd.read_csv('train.csv')

In [12]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [21]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [19]:
(df.isnull().mean()*100).round(2)

PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64

In [20]:
# Percent mein
null_pct = (df.isnull().mean() * 100).round(2).sort_values(ascending=False)
print(null_pct[null_pct > 0])

# Count + percent ek saath
pd.DataFrame({"count": df.isnull().sum(), "percent": df.isnull().mean() * 100})

Cabin       77.10
Age         19.87
Embarked     0.22
dtype: float64


,count,percent
PassengerId,0,0.000000
Survived,0,0.000000
Pclass,0,0.000000
Name,0,0.000000
Sex,0,0.000000
Age,177,19.865320
SibSp,0,0.000000
Parch,0,0.000000
Ticket,0,0.000000
Fare,0,0.000000


In [23]:
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

In [24]:
df["Has_Cabin"] = df["Cabin"].notnull().astype(int)

In [26]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Has_Cabin
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S,1
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",0,27.0,0,0,211536,13.0000,NaN,S,0
887,888,1,1,"Graham, Miss. Margaret Edith",1,19.0,0,0,112053,30.0000,B42,S,1
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",1,NaN,1,2,W./C. 6607,23.4500,NaN,S,0
889,890,1,1,"Behr, Mr. Karl Howell",0,26.0,0,0,111369,30.0000,C148,C,1


In [29]:
df['Cabin'].isnull().sum()

687

In [28]:

df['Has_Cabin'].value_counts()

Has_Cabin
0    687
1    204
Name: count, dtype: int64

In [30]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Has_Cabin"]
X = df[features]
y = df["Survived"]

In [31]:
X

,Pclass,Sex,Age,SibSp,Parch,Fare,Has_Cabin
0,3,0,22.0,1,0,7.2500,0
1,1,1,38.0,1,0,71.2833,1
2,3,1,26.0,0,0,7.9250,0
3,1,1,35.0,1,0,53.1000,1
4,3,0,35.0,0,0,8.0500,0
...,...,...,...,...,...,...,...
886,2,0,27.0,0,0,13.0000,0
887,1,1,19.0,0,0,30.0000,1
888,3,1,NaN,1,2,23.4500,0
889,1,0,26.0,0,0,30.0000,1


In [32]:
y

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64

In [33]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [34]:
imputers = {
    "Median": SimpleImputer(strategy="median"),
    "KNN": KNNImputer(n_neighbors=5),
    "MICE": IterativeImputer(max_iter=10, random_state=0),
}
models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(random_state=0),
}

In [35]:
results = {}
for i_name, imp in imputers.items():
    for m_name, model in models.items():
        pipe = make_pipeline(imp, StandardScaler(), model)
        scores = cross_val_score(pipe, X, y, cv=5)
        results[(i_name, m_name)] = scores.mean()
        print(f"{i_name:7} + {m_name:12} -> {scores.mean():.4f} (±{scores.std():.4f})")

Median  + Logistic     -> 0.7935 (±0.0173)
Median  + RandomForest -> 0.8182 (±0.0426)
KNN     + Logistic     -> 0.7901 (±0.0142)
KNN     + RandomForest -> 0.8003 (±0.0335)
MICE    + Logistic     -> 0.7980 (±0.0102)
MICE    + RandomForest -> 0.8126 (±0.0359)


In [37]:
from sklearn.ensemble import HistGradientBoostingClassifier

scores = cross_val_score(HistGradientBoostingClassifier(random_state=0), X, y, cv=5)
print(f"NaN as-is + HistGB -> {scores.mean():.4f} (±{scores.std():.4f})")

NaN as-is + HistGB -> 0.8339 (±0.0314)


In [38]:
from sklearn.ensemble import HistGradientBoostingClassifier

final_model = HistGradientBoostingClassifier(random_state=0)
final_model.fit(X, y)

HistGradientBoostingClassifier(random_state=0)

In [40]:
test = pd.read_csv("test.csv")

# Wahi cleaning (Step 2 jaisi)
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})
test["Has_Cabin"] = test["Cabin"].notnull().astype(int)

# Wahi features, wahi order
X_test = test[features]

# Predict
preds = final_model.predict(X_test)

In [42]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": preds
})
submission.to_csv("submission.csv", index=False)

In [43]:
print(submission.shape)                       # (418, 2) aana chahiye
print(submission.isnull().sum())              # dono columns mein 0 nulls
print(submission["Survived"].value_counts())  # 0 aur 1 dono dikhne chahiye
print(submission.head())

(418, 2)
PassengerId    0
Survived       0
dtype: int64
Survived
0    274
1    144
Name: count, dtype: int64
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         1
4          896         0
